In [6]:
import pandas as pd
import os

In [9]:

# 1. Read the raw tab-separated file
df = pd.read_csv(
    "/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    quoting=3,            # QUOTE_NONE: messages contain stray " characters
    encoding="utf-8",     # use "latin-1" if you get a UnicodeDecodeError
)

# 2. Basic cleanup
df["label"] = df["label"].str.strip().str.lower()
df["message"] = df["message"].astype(str).str.strip()
df = df.dropna(subset=["label", "message"])
df = df[df["label"].isin(["ham", "spam"])]

# 3. Helpful columns for analysis
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})
df["char_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()

# 4. Inspect before saving
print(df.shape)
print(df["label"].value_counts())
print("Duplicates:", df.duplicated(subset="message").sum())

# 5. Save as CSV
df.to_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/sms_spam.csv", index=False, encoding="utf-8")

(5574, 5)
label
ham     4827
spam     747
Name: count, dtype: int64
Duplicates: 414


## import and settings

In [10]:
import re
import html
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42

# one-time downloads (safe to re-run)
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

In [11]:
# load the data
df = pd.read_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/sms_spam.csv")

print("Shape:", df.shape)
df.head(10)

Shape: (5574, 5)


,label,message,label_num,char_count,word_count
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0,111,20
1,ham,Ok lar... Joking wif u oni...,0,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,1,155,28
3,ham,U dun say so early hor... U c already then say...,0,49,11
4,ham,"Nah I don't think he goes to usf, he lives around here though",0,61,13
5,spam,FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX...,1,147,32
6,ham,Even my brother is not like to speak with me. They treat me like aids patent.,0,77,16
7,ham,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Calle...,0,160,26
8,spam,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 0906170146...,1,157,26
9,spam,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call Th...,1,154,29


In [12]:
df.info()
print("\nMissing values:\n", df.isna().sum())
print("\nClass counts:\n", df["label"].value_counts())
print("\nClass %:\n", (df["label"].value_counts(normalize=True) * 100).round(2))

<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   label       5574 non-null   str  
 1   message     5574 non-null   str  
 2   label_num   5574 non-null   int64
 3   char_count  5574 non-null   int64
 4   word_count  5574 non-null   int64
dtypes: int64(3), str(2)
memory usage: 673.5 KB

Missing values:
 label         0
message       0
label_num     0
char_count    0
word_count    0
dtype: int64

Class counts:
 label
ham     4827
spam     747
Name: count, dtype: int64

Class %:
 label
ham     86.6
spam    13.4
Name: proportion, dtype: float64


## Data quality checks and cleaning

In [13]:
# Duplicates
dup_total = df.duplicated(subset="message").sum()
print("Duplicate messages:", dup_total)

# Do any duplicates have conflicting labels?
conflicts = (
    df.groupby("message")["label"].nunique()
      .loc[lambda s: s > 1]
)
print("Messages with conflicting labels:", len(conflicts))

# Most repeated messages
df["message"].value_counts().head(10)

Duplicate messages: 414
Messages with conflicting labels: 0


message
Sorry, I'll call later                                                                                                                                                                 30
I cant pick the phone right now. Pls send a message                                                                                                                                    12
Ok...                                                                                                                                                                                  10
Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £5000 prize!                                     4
Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care n luv!.. kettoda manda... Have nice day da.     4
Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 

In [14]:
# Remove exact duplicates (keeps the first occurrence)
df_clean = df.drop_duplicates(subset="message").reset_index(drop=True)

print("Before:", len(df), "| After:", len(df_clean))
print(df_clean["label"].value_counts())

Before: 5574 | After: 5160
label
ham     4518
spam     642
Name: count, dtype: int64


In [18]:
os.makedirs("data/processed", exist_ok=True)
df_clean.to_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/processed/sms_dedup.csv", index=False)